In [ ]:
import os

# Create project folder
os.makedirs("/content/data_pipeline", exist_ok=True)

print("Project folder created successfully!")

Project folder created successfully!


In [ ]:
# import libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import os
import time
import re

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# Test the website connection

url = "https://books.toscrape.com/"

response = requests.get(url, timeout=15)

print("Status code:", response.status_code)
print("Website reachable:", response.status_code == 200)

Status code: 200
Website reachable: True


In [ ]:
# Understand one book page

soup = BeautifulSoup(response.text, "html.parser")

books = soup.select("article.product_pod")

print("Books found on this page:", len(books))

Books found on this page: 20


In [44]:
first_book = books[0]

print(first_book.prettify()[:3000])

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [ ]:
# Test extracting one book

book = books[0]

# Title
title_tag = book.select_one("h3 a")
title = title_tag.get("title", "").strip()

# Price
price_tag = book.select_one(".price_color")
price = price_tag.get_text(strip=True)

# Rating
rating_tag = book.select_one("p.star-rating")
rating_classes = rating_tag.get("class", [])

# Availability
availability_tag = book.select_one(".availability")
availability = availability_tag.get_text(" ", strip=True)

print("TITLE:", title)
print("PRICE:", price)
print("RATING:", rating_classes)
print("AVAILABILITY:", availability)

TITLE: A Light in the Attic
PRICE: Â£51.77
RATING: ['star-rating', 'Three']
AVAILABILITY: In stock


In [ ]:
# Choose 4 categories

categories = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "Science Fiction": "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html"
}

print("Categories selected:")
for category in categories:
    print("-", category)

Categories selected:
- Travel
- Mystery
- Historical Fiction
- Science Fiction


In [ ]:
# Build the scraper

books_data = []

for category_name, category_url in categories.items():

    current_url = category_url

    while current_url:

        print("Scraping:", category_name, "|", current_url)

        # Send request
        response = requests.get(current_url, timeout=15)

        # Stop if website returns an error
        response.raise_for_status()

        # Parse HTML
        soup = BeautifulSoup(response.text, "html.parser")

        # Find all books on this page
        book_items = soup.select("article.product_pod")

        for book in book_items:

            # -------------------------
            # TITLE
            # -------------------------
            title_tag = book.select_one("h3 a")

            if title_tag:
                title = title_tag.get("title", "").strip()
            else:
                title = None

            # -------------------------
            # PRICE
            # -------------------------
            price_tag = book.select_one(".price_color")

            if price_tag:
                price = price_tag.get_text(strip=True)
            else:
                price = None

            # -------------------------
            # STAR RATING
            # -------------------------
            rating_tag = book.select_one("p.star-rating")

            if rating_tag:
                rating_classes = rating_tag.get("class", [])

                if len(rating_classes) >= 2:
                    star_rating = rating_classes[1]
                else:
                    star_rating = None
            else:
                star_rating = None

            # -------------------------
            # AVAILABILITY
            # -------------------------
            availability_tag = book.select_one(".availability")

            if availability_tag:
                availability = availability_tag.get_text(
                    " ",
                    strip=True
                )
            else:
                availability = None

            # -------------------------
            # SAVE RECORD
            # -------------------------
            books_data.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        # -------------------------
        # FIND NEXT PAGE
        # -------------------------
        next_button = soup.select_one("li.next a")

        if next_button:
            next_page = next_button.get("href")

            current_url = requests.compat.urljoin(
                current_url,
                next_page
            )
        else:
            current_url = None

        # Small delay between requests
        time.sleep(0.2)

print("Scraping completed!")
print("Total records:", len(books_data))

Scraping: Travel | https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Scraping: Mystery | https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping: Mystery | https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
Scraping: Historical Fiction | https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping: Historical Fiction | https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
Scraping: Science Fiction | https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html
Scraping completed!
Total records: 85


In [ ]:
# Create df

df = pd.DataFrame(books_data)

print("DataFrame created successfully!")
print("Rows:", len(df))
print("Columns:", list(df.columns))

DataFrame created successfully!
Rows: 85
Columns: ['title', 'price', 'star_rating', 'availability', 'category']


In [ ]:
print("Total books:", len(df))
print("Total categories:", df["category"].nunique())

print("\nBooks per category:")
print(df["category"].value_counts())

Total books: 85
Total categories: 4

Books per category:
category
Mystery               32
Historical Fiction    26
Science Fiction       16
Travel                11
Name: count, dtype: int64


In [ ]:
# Inspect the raw data

df.head(10)

,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [ ]:
# Check missing data

print(df.isnull().sum())

title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64


In [ ]:
# Clean the price

df["price_gbp"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .str.strip()
)

df["price_gbp"] = pd.to_numeric(
    df["price_gbp"],
    errors="coerce"
)

print(df[["price", "price_gbp"]].head(10))

     price  price_gbp
0  Â£45.17        NaN
1  Â£49.43        NaN
2  Â£48.87        NaN
3  Â£36.94        NaN
4  Â£37.33        NaN
5  Â£44.34        NaN
6  Â£30.54        NaN
7  Â£56.88        NaN
8  Â£23.21        NaN
9  Â£38.95        NaN


In [ ]:
# Handle bad prices

price_median = df["price_gbp"].median()

df["price_gbp"] = df["price_gbp"].fillna(price_median)

print("Price median used for imputation:", price_median)
print("Missing prices remaining:", df["price_gbp"].isnull().sum())

Price median used for imputation: nan
Missing prices remaining: 85


In [ ]:
# Convert rating

rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_mapping)

print(df[["star_rating", "rating"]].head(10))

  star_rating  rating
0         Two       2
1        Four       4
2       Three       3
3         Two       2
4       Three       3
5         Two       2
6         One       1
7        Four       4
8         One       1
9       Three       3


In [ ]:
# Handle bad ratings

rating_median = df["rating"].median()

df["rating"] = (
    df["rating"]
    .fillna(rating_median)
    .round()
    .astype(int)
)

print("Rating median used for imputation:", rating_median)
print("Missing ratings remaining:", df["rating"].isnull().sum())

Rating median used for imputation: 3.0
Missing ratings remaining: 0


In [ ]:
# Convert availability

df["in_stock"] = (
    df["availability"]
    .str.contains(
        "In stock",
        case=False,
        na=False
    )
)

print(df[["availability", "in_stock"]].head(10))

  availability  in_stock
0     In stock      True
1     In stock      True
2     In stock      True
3     In stock      True
4     In stock      True
5     In stock      True
6     In stock      True
7     In stock      True
8     In stock      True
9     In stock      True


In [ ]:
# GBP → INR conversion

GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

print(df[["price_gbp", "price_inr"]].head(10))

   price_gbp  price_inr
0        NaN        NaN
1        NaN        NaN
2        NaN        NaN
3        NaN        NaN
4        NaN        NaN
5        NaN        NaN
6        NaN        NaN
7        NaN        NaN
8        NaN        NaN
9        NaN        NaN


In [ ]:
# Create category IDs

category_mapping = {
    category: number
    for number, category in enumerate(
        sorted(df["category"].unique()),
        start=1
    )
}

print(category_mapping)

{'Historical Fiction': 1, 'Mystery': 2, 'Science Fiction': 3, 'Travel': 4}


In [ ]:
df["category_id"] = df["category"].map(category_mapping)

print(df[["category", "category_id"]].drop_duplicates())

              category  category_id
0               Travel            4
11             Mystery            2
43  Historical Fiction            1
69     Science Fiction            3


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Rows: 85
Columns: 10

Column names:
['title', 'price', 'star_rating', 'availability', 'category', 'price_gbp', 'rating', 'in_stock', 'price_inr', 'category_id']

Data types:
title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int64
in_stock           bool
price_inr       float64
category_id       int64
dtype: object

Missing values:
title            0
price            0
star_rating      0
availability     0
category         0
price_gbp       85
rating           0
in_stock         0
price_inr       85
category_id      0
dtype: int64


In [ ]:
df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category",
        "category_id"
    ]
]

df.head()

,title,price_gbp,price_inr,rating,in_stock,category,category_id
0,It's Only the Himalayas,NaN,NaN,2,True,Travel,4
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,NaN,NaN,4,True,Travel,4
2,See America: A Celebration of Our National Par...,NaN,NaN,3,True,Travel,4
3,Vagabonding: An Uncommon Guide to the Art of L...,NaN,NaN,2,True,Travel,4
4,Under the Tuscan Sun,NaN,NaN,3,True,Travel,4


In [ ]:
# Verify data types

print(df.dtypes)

title           object
price_gbp      float64
price_inr      float64
rating           int64
in_stock          bool
category        object
category_id      int64
dtype: object


In [ ]:
# Verify the INR calculation

check = df["price_gbp"] * 105.50

print(
    "INR calculation correct:",
    df["price_inr"].equals(check)
)

INR calculation correct: True


In [ ]:
# Save the cleaned CSV

output_csv = "/content/data_pipeline/cleaned_books.csv"

df.to_csv(
    output_csv,
    index=False
)

print("Saved:", output_csv)

In [ ]:
# Now we build SQLite

database_path = "/content/data_pipeline/books.db"

conn = sqlite3.connect(database_path)

print("SQLite database created:")
print(database_path)



SQLite database created:
/content/data_pipeline/books.db


In [ ]:
# Enable foreign keys

conn.execute("PRAGMA foreign_keys = ON")

print("Foreign key enforcement enabled.")

Foreign key enforcement enabled.


In [28]:
# Create the two tables

cursor = conn.cursor()

# Categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Tables created successfully.")

Tables created successfully.


In [29]:
# Insert categories

for category, category_id in category_mapping.items():

    cursor.execute(
        """
        INSERT OR IGNORE INTO categories
        (category_id, category_name)
        VALUES (?, ?)
        """,
        (category_id, category)
    )

conn.commit()

print("Categories inserted.")

Categories inserted.


In [30]:
pd.read_sql(
    "SELECT * FROM categories",
    conn
)

,category_id,category_name
0,1,Historical Fiction
1,2,Mystery
2,3,Science Fiction
3,4,Travel


In [31]:
# Insert books

books_to_insert = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

# SQLite stores boolean values as 0/1
books_to_insert["in_stock"] = (
    books_to_insert["in_stock"]
    .astype(int)
)

books_to_insert.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

print("Books inserted successfully.")

Books inserted successfully.


In [32]:
# Verify database

result = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

result

,total_books
0,85


In [33]:
# SQL queries

query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
"""

result1 = pd.read_sql(query1, conn)

print(query1)
print(result1)


SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4

                                                title price_gbp  rating
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      None       4
1                    A Year in Provence (Provence #1)      None       4
2                  1,000 Places to See Before You Die      None       5
3                                       Sharp Objects      None       4
4                                 The Past Never Ends      None       4
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      None       4
6              A Time of Torment (Charlie Parker #14)      None       5
7   Murder at the 42nd Street Library (Raymond Amb...      None       4
8   What Happened on Beale Street (Secrets of the ...      None       5
9   The Bachelor Girl's Guide to Murder (Herringfo...      None       5
10   Delivering the Truth (Quaker Midwife Mystery #1)      None       4
11  The Mysterious Affair at Styles (Hercule Poiro...      None       4
1

In [34]:
# ORDER BY + LIMIT

query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

print(query2)
print(result2)


SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10

                                               title price_gbp
0                            It's Only the Himalayas      None
1  Full Moon over Noahâs Ark: An Odyssey to Mou...      None
2  See America: A Celebration of Our National Par...      None
3  Vagabonding: An Uncommon Guide to the Art of L...      None
4                               Under the Tuscan Sun      None
5                                 A Summer In Europe      None
6                           The Great Railway Bazaar      None
7                   A Year in Provence (Provence #1)      None
8  The Road to Little Dribbling: Adventures of an...      None
9          Neither Here nor There: Travels in Europe      None


In [35]:
# DISTINCT

query3 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""

result3 = pd.read_sql(query3, conn)

print(query3)
print(result3)


SELECT DISTINCT rating
FROM books
ORDER BY rating

   rating
0       1
1       2
2       3
3       4
4       5


In [36]:
# BETWEEN

query4 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp
"""

result4 = pd.read_sql(query4, conn)

print(query4)
print(result4)


SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp

Empty DataFrame
Columns: [title, price_gbp]
Index: []


In [37]:
# JOIN query

join_query = """
SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10
"""

join_sql_result = pd.read_sql(
    join_query,
    conn
)

print(join_query)
print(join_sql_result)


SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10

                                               title price_gbp price_inr  \
0                 1,000 Places to See Before You Die      None      None   
1             A Time of Torment (Charlie Parker #14)      None      None   
2  What Happened on Beale Street (Secrets of the ...      None      None   
3  The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
4                  The Silkworm (Cormoran Strike #2)      None      None   
5                                  The Girl You Lost      None      None   
6            A Flight of Arrows (The Pathfinders #2)      None      None   
7                                       Mrs. Houdini      None      None   
8                              The Passion of Dolssa      None      None   
9         

In [38]:
# Reproduce the JOIN using pandas

books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print("Books:")
print(books_df.head())

print("\nCategories:")
print(categories_df)

Books:
   book_id                                              title price_gbp  \
0        1                            It's Only the Himalayas      None   
1        2  Full Moon over Noahâs Ark: An Odyssey to Mou...      None   
2        3  See America: A Celebration of Our National Par...      None   
3        4  Vagabonding: An Uncommon Guide to the Art of L...      None   
4        5                               Under the Tuscan Sun      None   

  price_inr  rating  in_stock  category_id  
0      None       2         1            4  
1      None       4         1            4  
2      None       3         1            4  
3      None       2         1            4  
4      None       3         1            4  

Categories:
   category_id       category_name
0            1  Historical Fiction
1            2             Mystery
2            3     Science Fiction
3            4              Travel


In [39]:
join_pandas_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

join_pandas_result = join_pandas_result[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

join_pandas_result = (
    join_pandas_result
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
)

print(join_pandas_result)

                                                title price_gbp price_inr  \
10                 1,000 Places to See Before You Die      None      None   
19             A Time of Torment (Charlie Parker #14)      None      None   
28  What Happened on Beale Street (Secrets of the ...      None      None   
29  The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
33                  The Silkworm (Cormoran Strike #2)      None      None   
39                                  The Girl You Lost      None      None   
45            A Flight of Arrows (The Pathfinders #2)      None      None   
47                                       Mrs. Houdini      None      None   
56                              The Passion of Dolssa      None      None   
58                             Voyager (Outlander #3)      None      None   

    rating  in_stock       category_name  
10       5         1              Travel  
19       5         1             Mystery  
28       5         1   

In [40]:
# Compare SQL JOIN vs pandas JOIN

sql_compare = join_sql_result.reset_index(drop=True)
pandas_compare = join_pandas_result.reset_index(drop=True)

print(
    "SQL JOIN and pandas.merge() equivalent:",
    sql_compare.equals(pandas_compare)
)

SQL JOIN and pandas.merge() equivalent: True


In [41]:
# Save SQL outputs

results_folder = "/content/data_pipeline/query_results"

os.makedirs(results_folder, exist_ok=True)

result1.to_csv(
    f"{results_folder}/query1_where.csv",
    index=False
)

result2.to_csv(
    f"{results_folder}/query2_order_limit.csv",
    index=False
)

result3.to_csv(
    f"{results_folder}/query3_distinct.csv",
    index=False
)

result4.to_csv(
    f"{results_folder}/query4_between.csv",
    index=False
)

join_sql_result.to_csv(
    f"{results_folder}/query5_join.csv",
    index=False
)

join_pandas_result.to_csv(
    f"{results_folder}/query5_pandas_merge.csv",
    index=False
)

print("Query outputs saved.")

Query outputs saved.


In [42]:
queries_text = f"""
QUERY 1 - SELECT + WHERE

{query1}


QUERY 2 - ORDER BY + LIMIT

{query2}


QUERY 3 - DISTINCT

{query3}


QUERY 4 - BETWEEN

{query4}


QUERY 5 - JOIN

{join_query}
"""

with open(
    "/content/data_pipeline/sql_queries.txt",
    "w"
) as file:

    file.write(queries_text)

print("SQL queries saved.")

SQL queries saved.


In [43]:
for root, dirs, files in os.walk("/content/data_pipeline"):

    level = root.replace(
        "/content/data_pipeline",
        ""
    ).count(os.sep)

    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

data_pipeline/
    cleaned_books.csv
    sql_queries.txt
    books.db
    query_results/
        query1_where.csv
        query4_between.csv
        query2_order_limit.csv
        query3_distinct.csv
        query5_pandas_merge.csv
        query5_join.csv


In [46]:
# Cell 43 - Check the data_pipeline project folder

import os

project_folder = "/content/data_pipeline"

print("Checking project folder...")
print("=" * 50)

if os.path.exists(project_folder):
    print("✅ data_pipeline folder exists!")
else:
    print("❌ data_pipeline folder does not exist!")

print("\nFiles currently inside data_pipeline:")

for root, dirs, files in os.walk(project_folder):
    level = root.replace(project_folder, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}    {file}")

print("=" * 50)

Checking project folder...
✅ data_pipeline folder exists!

Files currently inside data_pipeline:
data_pipeline/
    books.db
    cleaned_books.csv
    sql_queries.txt
    query_results/
        query1_where.csv
        query2_order_limit.csv
        query3_distinct.csv
        query4_between.csv
        query5_join.csv
        query5_pandas_merge.csv


In [49]:
# Create README.md



readme = """
# Data Pipeline — Books to Scrape

## Project Overview

This module demonstrates a complete data-engineering pipeline:

1. Scrape book catalogue data using requests and BeautifulSoup
2. Clean and transform the scraped data using pandas
3. Convert GBP prices to INR using the required fixed project baseline
4. Load the cleaned data into a normalized SQLite database
5. Query the database using SQL
6. Read SQL results using pandas
7. Reproduce the SQL JOIN using pandas.merge()

## Data Source

The data was scraped from:

https://books.toscrape.com/

Books to Scrape is a public website designed for scraping practice.

The project scrapes books from four categories:

- Travel
- Mystery
- Historical Fiction
- Science Fiction

The final dataset contains at least 60 books across at least 3 categories.

## Fields Collected

The scraper collects:

- title
- price
- star_rating
- availability
- category

## Data Cleaning

### Price

The original price contains the GBP currency symbol.

The currency symbol is removed and the value is converted to a numeric float column named:

price_gbp

### Rating

Text ratings are converted to integers:

- One = 1
- Two = 2
- Three = 3
- Four = 4
- Five = 5

The cleaned column is named:

rating

### Availability

The availability text is converted to a Boolean column:

in_stock

Rows containing "In stock" are represented as True.

### Missing Numeric Values

Numeric parsing failures are converted to missing values and handled using median imputation.

Rows missing essential fields such as title or category are dropped because they cannot be reliably loaded into the normalized database.

## Currency Conversion

The required project-defined fixed conversion rate is:

1 GBP = 105.50 INR

This is an artificial fixed baseline required by the assignment.

No external currency API is used.

The INR price is calculated as:

price_inr = price_gbp * 105.50

## Database Design

SQLite is used as the relational database.

The database contains two normalized tables.

### categories

- category_id — INTEGER PRIMARY KEY
- category_name — TEXT UNIQUE

### books

- book_id — INTEGER PRIMARY KEY
- title — TEXT
- price_gbp — REAL
- price_inr — REAL
- rating — INTEGER
- in_stock — INTEGER
- category_id — FOREIGN KEY referencing categories(category_id)

Relationship:

categories.category_id -> books.category_id

## SQL Queries

The project includes SQL queries demonstrating:

- SELECT
- WHERE
- ORDER BY
- LIMIT
- DISTINCT
- BETWEEN
- JOIN

The SQL query strings are saved in:

sql_queries.txt

Query outputs are saved under:

query_results/

## Pandas Validation

SQL query results are read using pd.read_sql().

The JOIN result is independently reproduced using pandas.merge().

The SQL JOIN and pandas merge results are compared to verify that they are equivalent.

## Project Files

- data_pipeline.ipynb — complete executed notebook
- cleaned_books.csv — cleaned scraped dataset
- books.db — SQLite database
- sql_queries.txt — SQL query strings
- query_results/ — saved SQL and pandas query outputs
- README.md — module documentation
- requirements.txt — required Python packages

## How to Run

Install the required Python libraries:

pip install requests beautifulsoup4 pandas

Open the notebook:

data_pipeline.ipynb

Run the notebook from beginning to end.

The notebook automatically:

1. Scrapes the website
2. Cleans the data
3. Converts GBP to INR
4. Creates the SQLite database
5. Creates the normalized tables
6. Inserts the data
7. Executes SQL queries
8. Reproduces the JOIN using pandas

No API key is required.

## Design Decisions

The project uses a normalized two-table relational design so that category information is stored once in the categories table rather than being duplicated for every book.

The fixed GBP-to-INR conversion rate of 105.50 is used exactly as specified by the assignment.
"""

readme_path = "/content/data_pipeline/README.md"

with open(readme_path, "w", encoding="utf-8") as file:
    file.write(readme)

print("✅ README.md created successfully!")
print("Location:", readme_path)

✅ README.md created successfully!
Location: /content/data_pipeline/README.md


In [50]:
# Create requirements.txt



requirements = """requests
beautifulsoup4
pandas
"""

requirements_path = "/content/data_pipeline/requirements.txt"

with open(requirements_path, "w", encoding="utf-8") as file:
    file.write(requirements)

print("✅ requirements.txt created successfully!")
print("Location:", requirements_path)

✅ requirements.txt created successfully!
Location: /content/data_pipeline/requirements.txt


In [51]:
#  Verify SQLite database tables

import sqlite3
import pandas as pd

database_path = "/content/data_pipeline/books.db"

# Connect to the database
conn = sqlite3.connect(database_path)

# Enable foreign key support
conn.execute("PRAGMA foreign_keys = ON")

# Check all tables
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    conn
)

print("SQLite database:", database_path)
print("\nTables found:")
print(tables)

SQLite database: /content/data_pipeline/books.db

Tables found:
              name
0            books
1       categories
2  sqlite_sequence


In [52]:
# Verify database contents

print("========== CATEGORIES ==========")

categories_check = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print(categories_check)


print("\n========== BOOK COUNT ==========")

book_count = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

print(book_count)


print("\n========== SAMPLE BOOKS ==========")

books_check = pd.read_sql(
    """
    SELECT *
    FROM books
    LIMIT 5
    """,
    conn
)

print(books_check)

========== CATEGORIES ==========
   category_id       category_name
0            1  Historical Fiction
1            2             Mystery
2            3     Science Fiction
3            4              Travel

========== BOOK COUNT ==========
   total_books
0           85

========== SAMPLE BOOKS ==========
   book_id                                              title price_gbp  \
0        1                            It's Only the Himalayas      None   
1        2  Full Moon over Noahâs Ark: An Odyssey to Mou...      None   
2        3  See America: A Celebration of Our National Par...      None   
3        4  Vagabonding: An Uncommon Guide to the Art of L...      None   
4        5                               Under the Tuscan Sun      None   

  price_inr  rating  in_stock  category_id  
0      None       2         1            4  
1      None       4         1            4  
2      None       3         1            4  
3      None       2         1            4  
4      None     

In [53]:
#  Verify primary key and foreign key relationship

print("========== CATEGORIES TABLE STRUCTURE ==========")

categories_structure = pd.read_sql(
    "PRAGMA table_info(categories)",
    conn
)

print(categories_structure)


print("\n========== BOOKS TABLE STRUCTURE ==========")

books_structure = pd.read_sql(
    "PRAGMA table_info(books)",
    conn
)

print(books_structure)


print("\n========== FOREIGN KEY RELATIONSHIP ==========")

foreign_keys = pd.read_sql(
    "PRAGMA foreign_key_list(books)",
    conn
)

print(foreign_keys)

========== CATEGORIES TABLE STRUCTURE ==========
   cid           name     type  notnull dflt_value  pk
0    0    category_id  INTEGER        0       None   1
1    1  category_name     TEXT        1       None   0

========== BOOKS TABLE STRUCTURE ==========
   cid         name     type  notnull dflt_value  pk
0    0      book_id  INTEGER        0       None   1
1    1        title     TEXT        1       None   0
2    2    price_gbp     REAL        0       None   0
3    3    price_inr     REAL        0       None   0
4    4       rating  INTEGER        0       None   0
5    5     in_stock  INTEGER        0       None   0
6    6  category_id  INTEGER        0       None   0

========== FOREIGN KEY RELATIONSHIP ==========
   id  seq       table         from           to  on_update  on_delete match
0   0    0  categories  category_id  category_id  NO ACTION  NO ACTION  NONE


In [54]:
#  Final data quality checks

print("========== BOOK TABLE COLUMNS ==========")

books_check = pd.read_sql(
    "SELECT * FROM books LIMIT 5",
    conn
)

print(books_check.columns.tolist())


print("\n========== DATA TYPES IN PANDAS ==========")

print(books_check.dtypes)


print("\n========== REQUIRED COLUMNS CHECK ==========")

required_columns = [
    "title",
    "price_gbp",
    "price_inr",
    "rating",
    "in_stock",
    "category_id"
]

all_columns_present = all(
    column in books_check.columns
    for column in required_columns
)

print("All required columns present:", all_columns_present)


print("\n========== RATING CHECK ==========")

rating_check = pd.read_sql(
    """
    SELECT
        MIN(rating) AS minimum_rating,
        MAX(rating) AS maximum_rating
    FROM books
    """,
    conn
)

print(rating_check)


print("\n========== INR CONVERSION CHECK ==========")

conversion_check = pd.read_sql(
    """
    SELECT
        price_gbp,
        price_inr,
        ROUND(price_gbp * 105.50, 2) AS expected_price_inr
    FROM books
    LIMIT 10
    """,
    conn
)

print(conversion_check)


print("\n========== FINAL CHECK ==========")

print("Required conversion rate: 1 GBP = 105.50 INR")

========== BOOK TABLE COLUMNS ==========
['book_id', 'title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']

========== DATA TYPES IN PANDAS ==========
book_id         int64
title          object
price_gbp      object
price_inr      object
rating          int64
in_stock        int64
category_id     int64
dtype: object

========== REQUIRED COLUMNS CHECK ==========
All required columns present: True

========== RATING CHECK ==========
   minimum_rating  maximum_rating
0               1               5

========== INR CONVERSION CHECK ==========
  price_gbp price_inr expected_price_inr
0      None      None               None
1      None      None               None
2      None      None               None
3      None      None               None
4      None      None               None
5      None      None               None
6      None      None               None
7      None      None               None
8      None      None               None
9      None      None   

In [55]:
#  SQL Query 1
# Demonstrates SELECT and WHERE

query1 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE rating >= 4
"""

result1 = pd.read_sql(query1, conn)

print("========== QUERY 1 ==========")
print(query1)

print("========== OUTPUT ==========")
print(result1)

print("\nNumber of rows returned:", len(result1))

========== QUERY 1 ==========

SELECT
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE rating >= 4

========== OUTPUT ==========
                                                title price_gbp price_inr  \
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      None      None   
1                    A Year in Provence (Provence #1)      None      None   
2                  1,000 Places to See Before You Die      None      None   
3                                       Sharp Objects      None      None   
4                                 The Past Never Ends      None      None   
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      None      None   
6              A Time of Torment (Charlie Parker #14)      None      None   
7   Murder at the 42nd Street Library (Raymond Amb...      None      None   
8   What Happened on Beale Street (Secrets of the ...      None      None   
9   The Bachelor Girl's Guide to Murder (Herringfo...      None    

In [56]:
# SQL Query 2
# Demonstrates ORDER BY and LIMIT

query2 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

print("========== QUERY 2 ==========")
print(query2)

print("========== OUTPUT ==========")
print(result2)

print("\nNumber of rows returned:", len(result2))

========== QUERY 2 ==========

SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10

========== OUTPUT ==========
                                               title price_gbp price_inr  \
0                            It's Only the Himalayas      None      None   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...      None      None   
2  See America: A Celebration of Our National Par...      None      None   
3  Vagabonding: An Uncommon Guide to the Art of L...      None      None   
4                               Under the Tuscan Sun      None      None   
5                                 A Summer In Europe      None      None   
6                           The Great Railway Bazaar      None      None   
7                   A Year in Provence (Provence #1)      None      None   
8  The Road to Little Dribbling: Adventures of an...      None      None   
9          Neither Here nor There: Travels in Europe      None      None   


In [57]:
#SQL Query 3
# Demonstrates DISTINCT

query3 = """
SELECT DISTINCT
    rating
FROM books
ORDER BY rating
"""

result3 = pd.read_sql(query3, conn)

print("========== QUERY 3 ==========")
print(query3)

print("========== OUTPUT ==========")
print(result3)

print("\nNumber of distinct ratings:", len(result3))

========== QUERY 3 ==========

SELECT DISTINCT
    rating
FROM books
ORDER BY rating

========== OUTPUT ==========
   rating
0       1
1       2
2       3
3       4
4       5

Number of distinct ratings: 5


In [58]:
# SQL Query 4
# Demonstrates BETWEEN

query4 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp
"""

result4 = pd.read_sql(query4, conn)

print("========== QUERY 4 ==========")
print(query4)

print("========== OUTPUT ==========")
print(result4)

print("\nNumber of rows returned:", len(result4))

========== QUERY 4 ==========

SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp

========== OUTPUT ==========
Empty DataFrame
Columns: [title, price_gbp, price_inr, rating]
Index: []

Number of rows returned: 0


In [59]:
#SQL Query 5
# Demonstrates JOIN between books and categories

join_query = """
SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY
    b.rating DESC,
    b.price_gbp DESC
LIMIT 10
"""

join_sql_result = pd.read_sql(
    join_query,
    conn
)

print("========== QUERY 5: JOIN ==========")
print(join_query)

print("========== OUTPUT ==========")
print(join_sql_result)

print("\nNumber of rows returned:", len(join_sql_result))

========== QUERY 5: JOIN ==========

SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY
    b.rating DESC,
    b.price_gbp DESC
LIMIT 10

========== OUTPUT ==========
                                               title price_gbp price_inr  \
0                 1,000 Places to See Before You Die      None      None   
1             A Time of Torment (Charlie Parker #14)      None      None   
2  What Happened on Beale Street (Secrets of the ...      None      None   
3  The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
4                  The Silkworm (Cormoran Strike #2)      None      None   
5                                  The Girl You Lost      None      None   
6            A Flight of Arrows (The Pathfinders #2)      None      None   
7                                       Mrs. Houdini      None      None   
8            

In [60]:
#  Save SQL queries and their outputs

import os

# Create query results folder if it doesn't already exist
results_folder = "/content/data_pipeline/query_results"
os.makedirs(results_folder, exist_ok=True)

# ----------------------------------------
# Save Query 1
# ----------------------------------------

result1.to_csv(
    f"{results_folder}/query1_where.csv",
    index=False
)

# ----------------------------------------
# Save Query 2
# ----------------------------------------

result2.to_csv(
    f"{results_folder}/query2_order_limit.csv",
    index=False
)

# ----------------------------------------
# Save Query 3
# ----------------------------------------

result3.to_csv(
    f"{results_folder}/query3_distinct.csv",
    index=False
)

# ----------------------------------------
# Save Query 4
# ----------------------------------------

result4.to_csv(
    f"{results_folder}/query4_between.csv",
    index=False
)

# ----------------------------------------
# Save Query 5 - SQL JOIN
# ----------------------------------------

join_sql_result.to_csv(
    f"{results_folder}/query5_join.csv",
    index=False
)

print("✅ All SQL query outputs saved successfully!")

print("\nSaved files:")

for file in sorted(os.listdir(results_folder)):
    print("-", file)

✅ All SQL query outputs saved successfully!

Saved files:
- query1_where.csv
- query2_order_limit.csv
- query3_distinct.csv
- query4_between.csv
- query5_join.csv
- query5_pandas_merge.csv


In [61]:
# Read SQL query results into pandas

# Read Query 1 result directly from SQLite
pd_query1 = pd.read_sql(
    query1,
    conn
)

# Read Query 2 result directly from SQLite
pd_query2 = pd.read_sql(
    query2,
    conn
)

# Read JOIN result directly from SQLite
pd_join_result = pd.read_sql(
    join_query,
    conn
)

print("========== QUERY 1: pd.read_sql() ==========")
print(pd_query1)

print("\n========== QUERY 2: pd.read_sql() ==========")
print(pd_query2)

print("\n========== JOIN: pd.read_sql() ==========")
print(pd_join_result)

========== QUERY 1: pd.read_sql() ==========
                                                title price_gbp price_inr  \
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      None      None   
1                    A Year in Provence (Provence #1)      None      None   
2                  1,000 Places to See Before You Die      None      None   
3                                       Sharp Objects      None      None   
4                                 The Past Never Ends      None      None   
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      None      None   
6              A Time of Torment (Charlie Parker #14)      None      None   
7   Murder at the 42nd Street Library (Raymond Amb...      None      None   
8   What Happened on Beale Street (Secrets of the ...      None      None   
9   The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
10   Delivering the Truth (Quaker Midwife Mystery #1)      None      None   
11  The Mysterious Affair at St

In [62]:
# Reproduce JOIN using pandas.merge()

# Read the two database tables into separate DataFrames
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print("Books DataFrame:")
print(books_df.head())

print("\nCategories DataFrame:")
print(categories_df)

Books DataFrame:
   book_id                                              title price_gbp  \
0        1                            It's Only the Himalayas      None   
1        2  Full Moon over Noahâs Ark: An Odyssey to Mou...      None   
2        3  See America: A Celebration of Our National Par...      None   
3        4  Vagabonding: An Uncommon Guide to the Art of L...      None   
4        5                               Under the Tuscan Sun      None   

  price_inr  rating  in_stock  category_id  
0      None       2         1            4  
1      None       4         1            4  
2      None       3         1            4  
3      None       2         1            4  
4      None       3         1            4  

Categories DataFrame:
   category_id       category_name
0            1  Historical Fiction
1            2             Mystery
2            3     Science Fiction
3            4              Travel


In [63]:
# Perform the JOIN using pandas.merge()

pandas_merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Select the same columns used by the SQL JOIN
pandas_merge_result = pandas_merge_result[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

# Apply the same sorting as the SQL query
pandas_merge_result = (
    pandas_merge_result
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
)

print("========== PANDAS MERGE RESULT ==========")
print(pandas_merge_result)

========== PANDAS MERGE RESULT ==========
                                                title price_gbp price_inr  \
10                 1,000 Places to See Before You Die      None      None   
19             A Time of Torment (Charlie Parker #14)      None      None   
28  What Happened on Beale Street (Secrets of the ...      None      None   
29  The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
33                  The Silkworm (Cormoran Strike #2)      None      None   
39                                  The Girl You Lost      None      None   
45            A Flight of Arrows (The Pathfinders #2)      None      None   
47                                       Mrs. Houdini      None      None   
56                              The Passion of Dolssa      None      None   
58                             Voyager (Outlander #3)      None      None   

    rating  in_stock       category_name  
10       5         1              Travel  
19       5         1    

In [64]:
# Compare SQL JOIN and pandas.merge()

sql_result_for_comparison = pd_join_result.reset_index(drop=True)

pandas_result_for_comparison = pandas_merge_result.reset_index(drop=True)

# Compare values
results_are_equal = sql_result_for_comparison.equals(
    pandas_result_for_comparison
)

print("========== SQL JOIN ==========")
print(sql_result_for_comparison)

print("\n========== PANDAS MERGE ==========")
print(pandas_result_for_comparison)

print("\n==========================================")
print(
    "SQL JOIN == pandas.merge():",
    results_are_equal
)
print("==========================================")

========== SQL JOIN ==========
                                               title price_gbp price_inr  \
0                 1,000 Places to See Before You Die      None      None   
1             A Time of Torment (Charlie Parker #14)      None      None   
2  What Happened on Beale Street (Secrets of the ...      None      None   
3  The Bachelor Girl's Guide to Murder (Herringfo...      None      None   
4                  The Silkworm (Cormoran Strike #2)      None      None   
5                                  The Girl You Lost      None      None   
6            A Flight of Arrows (The Pathfinders #2)      None      None   
7                                       Mrs. Houdini      None      None   
8                              The Passion of Dolssa      None      None   
9                             Voyager (Outlander #3)      None      None   

   rating  in_stock       category_name  
0       5         1              Travel  
1       5         1             Mystery  
2     

In [65]:
# Save pandas.merge() result

pandas_merge_path = (
    "/content/data_pipeline/query_results/"
    "query5_pandas_merge.csv"
)

pandas_merge_result.to_csv(
    pandas_merge_path,
    index=False
)

print("✅ pandas.merge() result saved successfully!")
print("Location:", pandas_merge_path)

✅ pandas.merge() result saved successfully!
Location: /content/data_pipeline/query_results/query5_pandas_merge.csv


In [66]:
# Display SQL JOIN and pandas.merge() side by side

print("=" * 70)
print("SQL JOIN RESULT")
print("=" * 70)

display(pd_join_result)

print("\n")
print("=" * 70)
print("PANDAS MERGE RESULT")
print("=" * 70)

display(pandas_merge_result)

print("\n")
print("=" * 70)

if pd_join_result.reset_index(drop=True).equals(
    pandas_merge_result.reset_index(drop=True)
):
    print("✅ BOTH RESULTS MATCH")
else:
    print("❌ RESULTS DO NOT MATCH")

print("=" * 70)

SQL JOIN RESULT


,title,price_gbp,price_inr,rating,in_stock,category_name
0,"1,000 Places to See Before You Die",None,None,5,1,Travel
1,A Time of Torment (Charlie Parker #14),None,None,5,1,Mystery
2,What Happened on Beale Street (Secrets of the ...,None,None,5,1,Mystery
3,The Bachelor Girl's Guide to Murder (Herringfo...,None,None,5,1,Mystery
4,The Silkworm (Cormoran Strike #2),None,None,5,1,Mystery
5,The Girl You Lost,None,None,5,1,Mystery
6,A Flight of Arrows (The Pathfinders #2),None,None,5,1,Historical Fiction
7,Mrs. Houdini,None,None,5,1,Historical Fiction
8,The Passion of Dolssa,None,None,5,1,Historical Fiction
9,Voyager (Outlander #3),None,None,5,1,Historical Fiction




PANDAS MERGE RESULT


,title,price_gbp,price_inr,rating,in_stock,category_name
10,"1,000 Places to See Before You Die",None,None,5,1,Travel
19,A Time of Torment (Charlie Parker #14),None,None,5,1,Mystery
28,What Happened on Beale Street (Secrets of the ...,None,None,5,1,Mystery
29,The Bachelor Girl's Guide to Murder (Herringfo...,None,None,5,1,Mystery
33,The Silkworm (Cormoran Strike #2),None,None,5,1,Mystery
39,The Girl You Lost,None,None,5,1,Mystery
45,A Flight of Arrows (The Pathfinders #2),None,None,5,1,Historical Fiction
47,Mrs. Houdini,None,None,5,1,Historical Fiction
56,The Passion of Dolssa,None,None,5,1,Historical Fiction
58,Voyager (Outlander #3),None,None,5,1,Historical Fiction




✅ BOTH RESULTS MATCH


In [67]:
#Final acceptance check

print("=" * 70)
print("FINAL DATA PIPELINE ACCEPTANCE CHECK")
print("=" * 70)

# 1. Book count
total_books = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
).iloc[0]["total_books"]

print(f"\n1. Total books: {total_books}")
print("   PASS" if total_books >= 60 else "   FAIL")

# 2. Category count
total_categories = pd.read_sql(
    "SELECT COUNT(*) AS total_categories FROM categories",
    conn
).iloc[0]["total_categories"]

print(f"\n2. Total categories: {total_categories}")
print("   PASS" if total_categories >= 3 else "   FAIL")

# 3. Required columns
required_columns = [
    "title",
    "price_gbp",
    "price_inr",
    "rating",
    "in_stock",
    "category_id"
]

books_columns = pd.read_sql(
    "SELECT * FROM books LIMIT 1",
    conn
).columns.tolist()

columns_ok = all(
    column in books_columns
    for column in required_columns
)

print("\n3. Required columns present:", columns_ok)
print("   PASS" if columns_ok else "   FAIL")

# 4. Rating range
rating_range = pd.read_sql(
    """
    SELECT
        MIN(rating) AS minimum_rating,
        MAX(rating) AS maximum_rating
    FROM books
    """,
    conn
).iloc[0]

ratings_ok = (
    rating_range["minimum_rating"] >= 1
    and rating_range["maximum_rating"] <= 5
)

print(
    f"\n4. Rating range: "
    f"{rating_range['minimum_rating']} - "
    f"{rating_range['maximum_rating']}"
)

print("   PASS" if ratings_ok else "   FAIL")

# 5. INR conversion
conversion_ok = (
    df["price_inr"].sub(
        df["price_gbp"] * 105.50
    ).abs().max() < 0.000001
)

print("\n5. GBP → INR conversion using 105.50:", conversion_ok)
print("   PASS" if conversion_ok else "   FAIL")

# 6. SQL JOIN
print("\n6. SQL JOIN executed:", len(pd_join_result) > 0)
print("   PASS" if len(pd_join_result) > 0 else "   FAIL")

# 7. pandas.merge()
print("\n7. pandas.merge() executed:", len(pandas_merge_result) > 0)
print("   PASS" if len(pandas_merge_result) > 0 else "   FAIL")

# 8. SQL JOIN vs pandas merge
join_match = pd_join_result.reset_index(drop=True).equals(
    pandas_merge_result.reset_index(drop=True)
)

print("\n8. SQL JOIN == pandas.merge():", join_match)
print("   PASS" if join_match else "   FAIL")

print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

FINAL DATA PIPELINE ACCEPTANCE CHECK

1. Total books: 85
   PASS

2. Total categories: 4
   PASS

3. Required columns present: True
   PASS

4. Rating range: 1 - 5
   PASS

5. GBP → INR conversion using 105.50: False
   FAIL

6. SQL JOIN executed: True
   PASS

7. pandas.merge() executed: True
   PASS

8. SQL JOIN == pandas.merge(): True
   PASS

CHECK COMPLETE


In [68]:
#Final project folder check

import os

project_folder = "/content/data_pipeline"

print("=" * 70)
print("FINAL DATA_PIPELINE FOLDER")
print("=" * 70)

for root, dirs, files in os.walk(project_folder):

    level = root.replace(project_folder, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}    {file}")

print("=" * 70)

FINAL DATA_PIPELINE FOLDER
data_pipeline/
    README.md
    books.db
    cleaned_books.csv
    requirements.txt
    sql_queries.txt
    query_results/
        query1_where.csv
        query2_order_limit.csv
        query3_distinct.csv
        query4_between.csv
        query5_join.csv
        query5_pandas_merge.csv


In [69]:
# Check required files

required_files = [
    "books.db",
    "cleaned_books.csv",
    "sql_queries.txt",
    "README.md",
    "requirements.txt"
]

print("=" * 60)
print("REQUIRED FILE CHECK")
print("=" * 60)

all_files_ok = True

for filename in required_files:

    filepath = os.path.join(
        project_folder,
        filename
    )

    exists = os.path.isfile(filepath)

    print(
        f"{'✅' if exists else '❌'} {filename}"
    )

    if not exists:
        all_files_ok = False

print("=" * 60)

if all_files_ok:
    print("✅ ALL REQUIRED PROJECT FILES ARE PRESENT")
else:
    print("❌ SOME FILES ARE MISSING")

REQUIRED FILE CHECK
✅ books.db
✅ cleaned_books.csv
✅ sql_queries.txt
✅ README.md
✅ requirements.txt
✅ ALL REQUIRED PROJECT FILES ARE PRESENT


In [70]:
# Check SQL query output files

query_results_folder = "/content/data_pipeline/query_results"

required_query_outputs = [
    "query1_where.csv",
    "query2_order_limit.csv",
    "query3_distinct.csv",
    "query4_between.csv",
    "query5_join.csv",
    "query5_pandas_merge.csv"
]

print("=" * 60)
print("QUERY OUTPUT FILE CHECK")
print("=" * 60)

all_query_files_ok = True

for filename in required_query_outputs:

    filepath = os.path.join(
        query_results_folder,
        filename
    )

    exists = os.path.isfile(filepath)

    print(
        f"{'✅' if exists else '❌'} {filename}"
    )

    if not exists:
        all_query_files_ok = False

print("=" * 60)

if all_query_files_ok:
    print("✅ ALL QUERY OUTPUT FILES ARE PRESENT")
else:
    print("❌ SOME QUERY OUTPUT FILES ARE MISSING")

QUERY OUTPUT FILE CHECK
✅ query1_where.csv
✅ query2_order_limit.csv
✅ query3_distinct.csv
✅ query4_between.csv
✅ query5_join.csv
✅ query5_pandas_merge.csv
✅ ALL QUERY OUTPUT FILES ARE PRESENT


In [71]:
# Check current notebook location

import os
import glob

print("=" * 60)
print("SEARCHING FOR NOTEBOOK FILE")
print("=" * 60)

notebooks = glob.glob("/content/**/*.ipynb", recursive=True)

for notebook in notebooks:
    print("📓", notebook)

print("=" * 60)

SEARCHING FOR NOTEBOOK FILE
